# Sanity checks — ERA5 diffusion downscaling

Visual checks for the build order in the README. Offline wiring is covered by
`python sanity_check.py`; this notebook is for the *visual* checks that need real
data / a trained model.

Run from the repo root after `data.download_era5` + `data.make_patches`.

In [ ]:
import sys; sys.path.insert(0, '..')
from pathlib import Path
import numpy as np, torch, matplotlib.pyplot as plt
from utils import load_config, get_device
from data.dataset import PatchDataset, load_norm_stats
from data.degrade import degrade, coarsen
cfg = load_config('config/default.yaml'); dev = get_device()
patch_dir = Path(cfg['paths']['patch_dir']); norm = load_norm_stats(patch_dir)
ds = PatchDataset(patch_dir/'test_patches.npy', norm); print(len(ds), 'test patches')

## Step 1 — degradation triplets
coarse → nearest-upsampled → ground truth should look right for 4x and 8x.

In [ ]:
hf = ds[0].unsqueeze(0)
fig, ax = plt.subplots(2, 3, figsize=(12, 8))
for row, ratio in enumerate([4, 8]):
    lo = coarsen(hf, ratio); up = degrade(hf, ratio)
    for a, img, ttl in zip(ax[row], [lo, up, hf], [f'coarse {ratio}x', 'nearest up', 'truth']):
        a.imshow(norm.decode(img)[0,0], cmap='RdBu_r'); a.set_title(ttl); a.axis('off')
plt.tight_layout()

## Step 2 — unconditional diffusion samples
After training, samples should resemble plausible Z500 fields.

In [ ]:
from sample.reconstruct import load_diffusion
model, diffusion, _ = load_diffusion(Path(cfg['paths']['ckpt_dir'])/'diffusion.pt', dev)
s = diffusion.sample_unconditional(model, (4,1,cfg['patches']['size'],cfg['patches']['size']), dev, n_steps=100)
s = norm.decode(s.cpu()).numpy()
fig, ax = plt.subplots(1,4, figsize=(16,4))
for a, img in zip(ax, s): a.imshow(img[0], cmap='RdBu_r'); a.axis('off')

## Step 3 — guided reconstruction beats bicubic (4x)

In [ ]:
from sample.reconstruct import reconstruct_diffusion, reconstruct_bicubic
rc = next(r for r in cfg['sample']['reconstructions'] if r['ratio']==4)
hf = ds[0].unsqueeze(0).to(dev)
diff = reconstruct_diffusion(diffusion, model, hf, 4, rc, eta=cfg['sample']['ddim_eta'], progress=True)
bic = reconstruct_bicubic(hf, 4)
fig, ax = plt.subplots(1,4, figsize=(18,4.5))
for a, t, ttl in zip(ax, [degrade(hf,4), bic, diff, hf], ['input','bicubic','diffusion','truth']):
    a.imshow(norm.decode(t.cpu())[0,0], cmap='RdBu_r'); a.set_title(ttl); a.axis('off')